In [ ]:
# =============================================================================
# Group-Level Speech/Pause Statistics and GLM Models
#
# PURPOSE:
# This notebook analyzes group differences between CWS and CWNS using
# participant-level speech, pause, and fluency-related metrics.
#
# INPUT:
# - CWNS_CWS_all_details.csv
#
# OUTPUTS:
# - Descriptive statistics by group
# - GLM / regression model results
# - p-value summary file: pval.csv
#
# ANALYSIS OVERVIEW:
# 1. Load participant-level speech and pause metrics
# 2. Rename columns for model compatibility
# 3. Convert selected duration measures from seconds to milliseconds
# 4. Define continuous and count-based outcome variables
# 5. Fit group models while accounting for sex and age
# 6. Save group-level p-values for later figure annotation
#
# NOTE:
# This notebook produces statistical outputs used by the results/figures notebook.
# =============================================================================

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

from scipy.stats import norm, shapiro, skew, kurtosis
from statsmodels.formula.api import ols, glm
from statsmodels.genmod.families import Poisson
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
# Load participant-level speech and pause dataset
# Update base_dir before running on a different machine

# Manually set the path to the directory where the file is located

# Project directory structure
#
# ATAS_CWNS_CWS_Project/        ← THIS is base_dir
# │
# ├── Audio_files_clipped/      ← create this folder; clipped audio files stored here
# ├── Visualize/                ← create this folder; .wav audio files for visualization/analysis stored here
# ├── Stat_csv_files/           ← input CSV files and summary statistics CSV files stored here
# ├── Individual_OutputCSV_Files/ ← individual output CSV files stored here
# │
# ├── Notebooks/                ← notebooks stored here
# │   ├── ATAS_Visualization/   ← visualization notebook
# │   └── ML_Analysis/          ← machine learning analysis notebooks
# │
# └── Scripts/                  ← function notebooks
#
base_dir = '..../ATAS_CWNS_CWS_Project'  # Change to required project directory

os.chdir(base_dir)  # Set the working directory
# Read the CSV file
xdata_pre = pd.read_csv(base_dir + "/Stat_csv_files/CWNS_CWS_all_details.csv") # Assuming data.csv exists in the specified directory


# Define output path for p-values used later in figure generation
# for storing the p-value file
file_path = base_dir + '/Stat_csv_files/pval.csv'

In [ ]:
# Rename columns to remove spaces and make names compatible with model formulas
xdata_pre = xdata_pre.rename(columns={
    'Mean Pause_s': 'Mean_Pause_s',
    'Mean Speech_s': 'Mean_Speech_s',
    'CV Pause': 'CV_Pause',
    'CV Speech': 'CV_Speech',
})
# Convert selected duration variables from seconds to milliseconds
# New columns are created so original second-based values are preserved

# List of columns to convert unit
columns_to_convert = ['Mean_Speech_s', 'Mean_Pause_s', 'long_p_durations_mean', 'short_p_durations_mean']

# Convert from seconds to milliseconds and create new columns
for col in columns_to_convert:
    xdata_pre[f"{col}_ms"] = xdata_pre[col] * 1000

In [ ]:
# Define outcome variables for statistical models
# Continuous metrics are modeled separately from count-based metrics

# Define variables for continuous and count models
continuous_metrics = {
    "Speech_Rate": "Speech Rate",
    "Pause_Duration_s": "Pause Duration (s)",
    "Mean_Pause_s_ms": "Mean Pause Duration (ms)",
    "Mean_Speech_s_ms": "Mean Speech Duration (ms)",
    "CV_Pause": "Coefficient of Variation - Pause",
    "CV_Speech": "Coefficient of Variation - Speech",
    "long_p_durations_mean_ms": "Mean Long Pause Duration (ms)",
    "short_p_durations_mean_ms": "Mean Short Pause Duration (ms)",
    "long_p_durations_cv": "CV of Long Pauses",
    "short_p_durations_cv": "CV of Short Pauses"
}

count_metrics = {
    "Pause_Events": "Number of Pause Events",
    "long_p_count": "Long Pause Events",
    "short_p_count": "Short Pause Events"
}
xdata = xdata_pre.copy()

In [ ]:
def custom_round(x):
    fractional = x - int(x)
    if fractional > 0.5:
        return int(x) + 1
    else:
        return int(x)
# Round participant ages for use as model covariates
# Apply to Age column
xdata['Age'] = xdata['Age'].apply(custom_round)

## Compute group-level descriptive statistics for each outcome variable

In [ ]:
# Define a function to calculate the statistics for continuous and count metrics
def calculate_statistics(df, group_col, metrics):
    # Create an empty list to hold the individual metric DataFrames
    metric_stats = []
    
    for metric, name in metrics.items():
        # Calculate the statistics for each metric in the group
        group_stats = df.groupby(group_col)[metric].agg(
            mean='mean',
            std='std',
            max='max',
            min='min'                       
        )
        # Format to 3 decimal places
        group_stats = group_stats.round(3)
        # Rename the columns to match the friendly names
        group_stats.columns = [f'{name}_mean',   f'{name}_std',f'{name}_max',f'{name}_min',]
        
        metric_stats.append(group_stats)
    
    # Concatenate all the metric stats into a single DataFrame
    return pd.concat(metric_stats, axis=1)

# Assume xdata is your dataframe and 'group_column' is the column for AWS and AWNS grouping
group_column = 'Group'  # Adjust this if the group column name is different in your data

# Calculate statistics for continuous metrics
continuous_stats = calculate_statistics(xdata, group_column, continuous_metrics)

# Calculate statistics for count metrics
count_stats = calculate_statistics(xdata, group_column, count_metrics)

# Combine both continuous and count statistics into a final table
final_table = pd.concat([continuous_stats, count_stats], axis=1)


In [ ]:
# # Adjust pandas display options to avoid clipping of the table
# pd.set_option('display.max_columns', None)  # Show all columns
# pd.set_option('display.max_rows', None)     # Show all rows (if the table is not too large)
# pd.set_option('display.width', None)        # Disable line wrapping
# pd.set_option('display.max_colwidth', None) # Disable column width truncation

# # Now, display the final_table
# final_table_T = final_table.T
# final_table_T


## Check model assumptions and predictor structure before fitting GLMs

In [ ]:
def check_glm_assumptions_stats(xdata, continuous_metrics, count_metrics):
    predictors = ["Group", "Sex", "Age"]
    
    # Convert categorical variables to dummies for VIF calculation
    X = xdata[predictors].copy()
    X = pd.get_dummies(X, drop_first=True)

    # 1. Compute VIF for Predictors
    print("\n### Checking Multicollinearity (VIF) ###")
    vif_data = pd.DataFrame()
    vif_data["Variable"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
    
    print(vif_data)
    print("\nInterpretation: VIF > 10 indicates severe multicollinearity. Consider removing/reducing correlated predictors.")

    # 2. Check Statistical Properties of Continuous Variables
    print("\n### Statistical Summary of Continuous Variables (Before and After Log Transformation) ###")
    
    for metric in continuous_metrics.keys():
        log_metric = f'log_{metric}'
        xdata[log_metric] = np.log1p(xdata[metric])  # Log transformation (if applicable)

        # Compute summary statistics before transformation
        mean_val_orig = np.mean(xdata[metric])
        std_val_orig = np.std(xdata[metric])
        skewness_orig = skew(xdata[metric])
        kurt_orig = kurtosis(xdata[metric])

        print(f"\n{metric} (Original):")
        print(f"Mean: {mean_val_orig:.4f}, Std Dev: {std_val_orig:.4f}, Skewness: {skewness_orig:.4f}, Kurtosis: {kurt_orig:.4f}")

        # Shapiro-Wilk Test before transformation
        shapiro_test_orig = shapiro(xdata[metric])
        print(f"Shapiro-Wilk Test (Original): W = {shapiro_test_orig.statistic:.4f}, p-value = {shapiro_test_orig.pvalue:.4f}")

        if shapiro_test_orig.pvalue < 0.05:
            print("  -> Data is NOT normally distributed (p < 0.05). Consider transformation.")
            
            # Compute summary statistics after transformation
            mean_val_log = np.mean(xdata[log_metric])
            std_val_log = np.std(xdata[log_metric])
            skewness_log = skew(xdata[log_metric])
            kurt_log = kurtosis(xdata[log_metric])

            print(f"\n{log_metric} (Log-Transformed):")
            print(f"Mean: {mean_val_log:.4f}, Std Dev: {std_val_log:.4f}, Skewness: {skewness_log:.4f}, Kurtosis: {kurt_log:.4f}")

            # Shapiro-Wilk Test after transformation
            shapiro_test_log = shapiro(xdata[log_metric])
            print(f"Shapiro-Wilk Test (Log-Transformed): W = {shapiro_test_log.statistic:.4f}, p-value = {shapiro_test_log.pvalue:.4f}")

            if shapiro_test_log.pvalue < 0.05:
                print("  -> Data is STILL not normally distributed (p < 0.05). Consider alternative transformations or robust methods.")
            else:
                print("  -> Log transformation improved normality (p > 0.05).")
        else:
            print("  -> Data appears normally distributed (p > 0.05). Skipping log-transformation test.")

    # 3. Breusch-Pagan Test for Homoscedasticity (Using Original Data)
    print("\n### Homoscedasticity Check (Breusch-Pagan Test) ###")

    for metric in continuous_metrics.keys():
        formula = f'{metric} ~ Group + Sex + Age'  # Use original variable, not log-transformed
        model = smf.glm(formula, data=xdata, family=sm.families.Gaussian()).fit()

        residuals = model.resid_deviance
        fitted_values = model.fittedvalues

        bp_test = het_breuschpagan(residuals, sm.add_constant(fitted_values))
        print(f"\n{metric} - Breusch-Pagan Test:")
        print(f"LM Statistic: {bp_test[0]:.4f}, p-value: {bp_test[1]:.4f}")

        if bp_test[1] < 0.05:
            print("  -> Residual variance is NOT constant (heteroscedasticity detected). Consider weighted regression.")
        else:
            print("  -> Residual variance appears constant (homoscedasticity).")
        
    print("\n### Checking Overdispersion in Poisson Models ###")
    for metric in count_metrics.keys():
        formula = f'{metric} ~ Group + Sex + Age'
        model = smf.glm(formula, data=xdata, family=sm.families.Poisson()).fit()
        
        pearson_chi2 = sum((model.resid_pearson)**2)
        df_residual = model.df_resid
        dispersion_ratio = pearson_chi2 / df_residual
        print(f"{metric}: Dispersion Ratio = {dispersion_ratio}")
        
        if dispersion_ratio > 1.5:
            print(f"WARNING: {metric} shows signs of overdispersion (Dispersion Ratio > 1.5). Consider using a Negative Binomial model.")

# Run statistical assumption checks
check_glm_assumptions_stats(xdata, continuous_metrics, count_metrics)


## Models for continuous and count-based ATAS metrics

In [ ]:
def model_continuous_metrics(xdata, continuous_metrics):
    predictors = ["Group", "Sex", "Age"]

    log_transform_map = {
        "Pause_Duration_s": "log_Pause_Duration_s",
        "Mean_Pause_s_ms": "log_Mean_Pause_s_ms",
        "long_p_durations_mean_ms": "log_long_p_durations_mean_ms",
        "long_p_durations_cv": "log_long_p_durations_cv",
    }

    gamma_glm_metrics = [
        "Mean_Speech_s_ms",
        "short_p_durations_cv",
    ]

    updated_metrics = {
        metric: metric
        for metric in continuous_metrics
        if metric not in gamma_glm_metrics
    }

    for orig, log_var in log_transform_map.items():
        if orig in updated_metrics:
            xdata[log_var] = np.where(xdata[orig] > 0, np.log(xdata[orig]), np.nan)
            updated_metrics[log_var] = updated_metrics.pop(orig)

    for metric in gamma_glm_metrics:
        updated_metrics[metric] = metric

    results = []

    for metric in updated_metrics:
        #print(metric)

        formula = f"{metric} ~ {' + '.join(predictors)}"
        model_data = xdata[[metric] + predictors].dropna()

        if model_data.empty:
            continue

        if metric in gamma_glm_metrics:
            model = smf.glm(
                formula,
                data=model_data,
                family=sm.families.Gamma(link=sm.families.links.log()),
            ).fit()
            model_type = "GLM (Gamma)"
        else:
            model = smf.glm(
                formula,
                data=model_data,
                family=sm.families.Gaussian(),
            ).fit()
            model_type = "GLM (Gaussian)"

        summary_df = pd.DataFrame({
            "Metric": [metric] * len(model.params),
            "Predictor": model.params.index,
            "Coefficient": model.params.values,
            "Std Error": model.bse.values,
            "z-value": model.tvalues.values,
            "p-value": model.pvalues.values,
            "Model Type": [model_type] * len(model.params),
        })

        results.append(summary_df)

    return pd.concat(results, ignore_index=True)


def fit_standard_negative_binomial(xdata, count_metrics):
    predictors = ["Group", "Sex", "Age"]
    results = []

    for metric in count_metrics.keys():
        y = pd.to_numeric(xdata[metric], errors="coerce")

        X = xdata[predictors].copy()
        X = pd.get_dummies(X, columns=["Group", "Sex"], drop_first=True)
        X = sm.add_constant(X)

        X = X.dropna()
        y = y.loc[X.index]

        if X.shape[0] == 0:
            print(f"Warning: No valid data available for {metric}. Skipping.")
            continue

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model = sm.NegativeBinomial(y, X).fit(disp=False)
        

        summary = model.summary2().tables[1]
        std_err_col = [col for col in summary.columns if "Std" in col][0]

        for index, row in summary.iterrows():
            results.append({
                "Metric": metric,
                "Predictor": index,
                "Coefficient": row["Coef."],
                "Std Error": row[std_err_col],
                "z-value": row["z"],
                "p-value": row["P>|z|"],
                "Model Type": "Negative Binomial",
            })

    results_df_count = pd.DataFrame(results)

    return results_df_count, X, y


# Run models
results_df = model_continuous_metrics(xdata, continuous_metrics)
results_df_count, X, y = fit_standard_negative_binomial(xdata, count_metrics)

# Combine results
all_results = pd.concat([results_df, results_df_count], ignore_index=True)


# Filter Group effect
all_results_filtered = all_results[
    (all_results["Predictor"] == "Group[T.CWS]") |
    (all_results["Predictor"] == "Group_CWS")
]


# Final p-value table for Group effect
final_results_df = (
    all_results_filtered[["Metric", "p-value"]]
    .copy()
    .sort_values(by="Metric")
    .reset_index(drop=True)
)



# Save final results
final_results_df.to_csv(file_path, index=False)
#print(f"CSV file saved at: {file_path}")

# Age effect
all_results_filtered_age = all_results[
    all_results["Predictor"] == "Age"
]


# Significant results
significant_results = all_results.loc[
    all_results["p-value"] < 0.05,
    ["Metric", "Predictor"]
]


In [ ]:
# Complete combined results from all models
# Includes coefficients, p-values, z-values, etc. for every predictor
# across continuous and count metrics
#print(all_results)

# Filtered table containing only Group effect p-values
# One row per metric
#print(final_results_df)

# Results corresponding only to the Age predictor
# across all fitted models
#print(all_results_filtered_age)

# Results corresponding only to Group effects
# (e.g., Group[T.CWS] or Group_CWS)
#print(all_results_filtered)

# Only statistically significant predictors
# where p-value < 0.05
#print(significant_results)

## Group-by-age interactions

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf


def summarize_model(model, metric, model_type):
    return pd.DataFrame({
        "Metric": [metric] * len(model.params),
        "Predictor": model.params.index,
        "Coefficient": model.params.values,
        "Std Error": model.bse.values,
        "z-value": model.tvalues.values,
        "p-value": model.pvalues.values,
        "Model Type": [model_type] * len(model.params)
    })


def model_continuous_metrics_with_interaction(xdata, continuous_metrics):
    predictors = ["Group", "Sex", "Age"]

    log_transform_map = {
        "Pause_Duration_s": "log_Pause_Duration_s",
        "Mean_Pause_s_ms": "log_Mean_Pause_s_ms",
        "long_p_durations_mean_ms": "log_long_p_durations_mean_ms",
        "long_p_durations_cv": "log_long_p_durations_cv"
    }

    gamma_glm_metrics = [
        "Mean_Speech_s_ms",
        "short_p_durations_cv"
    ]

    xdata = xdata.copy()

    updated_metrics = {
        metric: metric for metric in continuous_metrics
        if metric not in gamma_glm_metrics
    }

    for orig, log_var in log_transform_map.items():
        if orig in updated_metrics:
            xdata[log_var] = np.where(xdata[orig] > 0, np.log(xdata[orig]), np.nan)
            updated_metrics[log_var] = updated_metrics.pop(orig)

    for metric in gamma_glm_metrics:
        if metric in continuous_metrics:
            updated_metrics[metric] = metric

    results = []

    for metric in updated_metrics:
        print(metric)

        formula = f"{metric} ~ Group * Age + Sex"

        model_data = xdata[[metric, "Group", "Sex", "Age"]].dropna()
        if model_data.empty:
            continue

        if metric in gamma_glm_metrics:
            model = smf.glm(
                formula=formula,
                data=model_data,
                family=sm.families.Gamma(link=sm.families.links.log())
            ).fit()
            model_type = "GLM (Gamma)"
        else:
            model = smf.glm(
                formula=formula,
                data=model_data,
                family=sm.families.Gaussian()
            ).fit()
            model_type = "GLM (Gaussian)"

        results.append(summarize_model(model, metric, model_type))

    return pd.concat(results, ignore_index=True)


def fit_negative_binomial_with_interaction(xdata, count_metrics):
    results = []

    xdata = xdata.copy()

    for metric in count_metrics.keys():
        print(metric)

        formula = f"{metric} ~ Group * Age + Sex"

        model_data = xdata[[metric, "Group", "Sex", "Age"]].dropna()
        model_data[metric] = pd.to_numeric(model_data[metric], errors="coerce")
        model_data = model_data.dropna()

        if model_data.empty:
            print(f"Warning: No valid data available for {metric}. Skipping.")
            continue

        model = smf.negativebinomial(
            formula=formula,
            data=model_data
        ).fit(disp=False)

        results.append(summarize_model(model, metric, "Negative Binomial"))

    return pd.concat(results, ignore_index=True)


def get_group_age_interactions(results_df):
    return results_df[
        results_df["Predictor"].str.contains(
            "Group.*:Age|Age:Group",
            regex=True,
            na=False
        )
    ].copy()

In [ ]:
results_df_continuous_interaction = model_continuous_metrics_with_interaction(
    xdata,
    continuous_metrics
)

results_df_count_interaction = fit_negative_binomial_with_interaction(
    xdata,
    count_metrics
)

all_interaction_model_results = pd.concat(
    [results_df_continuous_interaction, results_df_count_interaction],
    ignore_index=True
)

group_age_interaction_results = get_group_age_interactions(
    all_interaction_model_results
)

group_age_interaction_results

## Benjamini-Hochberg correction 

In [ ]:
import numpy as np
import pandas as pd

p_values = all_results_filtered["p-value"].to_numpy()
metrics = all_results_filtered["Metric"].to_numpy()

def benjamini_hochberg(p_values, fdr=0.1):
    p_values = np.array(p_values)
    n = len(p_values)

    sorted_indices = np.argsort(p_values)
    sorted_p_values = p_values[sorted_indices]
    thresholds = fdr * np.arange(1, n + 1) / n

    reject = sorted_p_values <= thresholds

    if np.any(reject):
        max_idx = np.where(reject)[0].max()
        reject[:max_idx + 1] = True
    else:
        reject[:] = False

    return reject[np.argsort(sorted_indices)]

def bh_adjust(p_values):
    p_values = np.array(p_values)
    n = len(p_values)

    sorted_idx = np.argsort(p_values)
    sorted_p = p_values[sorted_idx]

    adjusted = np.zeros(n)

    for i in range(n):
        adjusted[i] = sorted_p[i] * n / (i + 1)

    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)

    return adjusted[np.argsort(sorted_idx)]

p_fdr = bh_adjust(p_values)
significant = benjamini_hochberg(p_values, fdr=0.1)

fdr_results = pd.DataFrame({
    "Metric": metrics,
    "p": p_values,
    "p_FDR": p_fdr,
    "FDR_sig": significant
})

#print(fdr_results)